In [1]:
#| default_exp perf.profiling
#| export

import sys
import os
import time
import numpy as np
import tracemalloc
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict
import gc

# Import from TinyTorch package (previous modules must be completed and exported)
from tinytorch.core.tensor import Tensor
from tinytorch.core.layers import Linear
from tinytorch.core.spatial import Conv2d

# Constants for memory and performance measurement
BYTES_PER_FLOAT32 = 4  # Standard float32 size in bytes
KB_TO_BYTES = 1024  # Kilobytes to bytes conversion
MB_TO_BYTES = 1024 * 1024  # Megabytes to bytes conversion

In [20]:
#| export
class Profiler:
    """
    Professional-grade ML model profiler for performance analysis.

    Measures parameters, FLOPs, memory usage, and latency with statistical rigor.
    Used for optimization guidance and deployment planning.
    """

    def __init__(self):
        """
        Initialize profiler with measurement state.

        TODO: Set up profiler tracking structures

        APPROACH:
        1. Create empty measurements dictionary
        2. Initialize operation counters
        3. Set up memory tracking state

        EXAMPLE:
        >>> profiler = Profiler()
        >>> profiler.measurements
        {}

        HINTS:
        - Use defaultdict(int) for operation counters
        - measurements dict will store timing results
        """
        ### BEGIN SOLUTION
        self.measurements = {}
        self.operation_counts = defaultdict(int)
        self.memory_tracker = None
        ### END SOLUTION

    def _count_layer_parameters(self, layer) -> int:
        """
        Count parameters in a single layer by inspecting weight and bias attributes.

        Handles the fundamental unit of parameter counting: a single layer
        with weight and optional bias tensors.

        ```
        Single Layer Parameter Count:
        ┌─────────────────────────────────────────┐
        │ layer.weight.data.size  (e.g., 128×64)  │
        │ + layer.bias.data.size  (e.g., 64)      │
        │ = total layer parameters (e.g., 8256)    │
        └─────────────────────────────────────────┘
        ```

        Args:
            layer: A layer object with .weight (and optionally .bias)

        Returns:
            int: Total parameter count for this layer
        """
        ### BEGIN SOLUTION
        params = 0
        if hasattr(layer, "weight"):
            params += layer.weight.data.size
            if hasattr(layer, 'bias') and layer.bias is not None:
                params += layer.bias.data.size
        return params
        ### END SOLUTION

    def count_parameters(self, model) -> int:
        """
        Count total trainable parameters in a model.

        TODO: Implement parameter counting for any model with parameters() method

        APPROACH:
        1. Get all parameters from model.parameters() if available
        2. For single layers, use _count_layer_parameters() helper
        3. Sum total element count across all parameter tensors

        EXAMPLE:
        >>> linear = Linear(128, 64)  # 128*64 + 64 = 8256 parameters
        >>> profiler = Profiler()
        >>> count = profiler.count_parameters(linear)
        >>> print(count)
        8256

        HINTS:
        - Use _count_layer_parameters() for single layers
        - Use parameter.data.size for tensor element count
        - Handle models with and without parameters() method
        """
        ### BEGIN SOLUTION
        if hasattr(model, 'layers'):
            return sum(p.data.size for layer in model.layers for p in layer.parameters())
        elif hasattr(model, 'parameters'):
            return sum(p.data.size for p in model.parameters())
        elif hasattr(model, 'weight'):
            return self._count_layer_parameters(model)
        return 0
        ### END SOLUTION

    def _count_linear_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs for a Linear layer forward pass.

        ```
        Linear FLOP Formula:
        FLOPs = in_features × out_features × 2
                     ↑              ↑          ↑
              Input dimension  Output dimension  Multiply + Add
        ```

        Args:
            model: A Linear layer with .weight attribute
            input_shape: Input tensor shape (batch, in_features)

        Returns:
            int: FLOP count for one forward pass (batch-independent)
        """
        ### BEGIN SOLUTION
        in_features = input_shape[-1]
        out_features = model.weight.shape[1] if hasattr(model, 'weight') else 1
        return in_features * out_features * 2
        ### END SOLUTION

    def _count_conv_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs for a Conv2d layer forward pass.

        ```
        Conv2d FLOP Formula:
        FLOPs = out_H × out_W × kernel_H × kernel_W × in_C × out_C × 2
                  ↑       ↑        ↑          ↑         ↑       ↑      ↑
              Output spatial    Kernel spatial     Channel dims   Mul+Add
        ```

        Args:
            model: A Conv2d layer with kernel_size, in_channels, out_channels
            input_shape: Input tensor shape (batch, channels, height, width)

        Returns:
            int: FLOP count for one forward pass
        """
        ### BEGIN SOLUTION
        if not (hasattr(model, 'kernel_size') and hasattr(model, 'in_channels')):
            return 0
        
        in_channels = model.in_channels
        out_channels = model.out_channels
        kernel_h = kernel_w = model.kernel_size

        input_h, input_w = input_shape[-2], input_shape[-1]
        stride = model.stride if hasattr(model, 'stride') else 1
        output_h = input_h // stride
        output_w = input_w // stride

        return output_h * output_w * kernel_h * kernel_w * in_channels * out_channels * 2
        ### END SOLUTION

    def _count_sequential_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs for a Sequential model by summing per-layer FLOPs.

        ```
        Sequential FLOP Accumulation:
        Layer 1 FLOPs + Layer 2 FLOPs + ... + Layer N FLOPs = Total FLOPs
             ↓               ↓                    ↓
          Shape propagated through each layer
        ```

        Args:
            model: A model with .layers attribute (list of layers)
            input_shape: Input tensor shape for the first layer

        Returns:
            int: Total FLOP count across all layers
        """
        ### BEGIN SOLUTION
        total_flops = 0
        current_shape = input_shape
        for layer in model.layers:
            total_flops += self.count_flops(layer, current_shape)
            if hasattr(layer, 'weight'):
                current_shape = current_shape[:-1] + (layer.weight.shape[1],)
        return total_flops
        ### END SOLUTION

    def count_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs (Floating Point Operations) for one forward pass.

        TODO: Implement FLOP counting by dispatching to per-layer-type helpers

        APPROACH:
        1. Identify model type by class name
        2. Dispatch to _count_linear_flops, _count_conv_flops, or _count_sequential_flops
        3. Fall back to 1 FLOP per element for activations

        EXAMPLE:
        >>> linear = Linear(128, 64)
        >>> profiler = Profiler()
        >>> flops = profiler.count_flops(linear, (1, 128))
        >>> print(flops)  # 128 * 64 * 2 = 16384
        16384

        HINT: Use model.__class__.__name__ to identify layer type
        """
        ### BEGIN SOLUTION
        model_name = model.__class__.__name__

        if model_name == 'Linear':
            return self._count_linear_flops(model, input_shape)
        elif model_name == 'Conv2d':
            return self._count_conv_flops(model, input_shape)
        elif model_name == 'Sequential' or hasattr(model, 'layers'):
            return self._count_sequential_flops(model, input_shape)
        else:
            return int(np.prod(input_shape))
        ### END SOLUTION

    def _calculate_parameter_memory(self, model) -> float:
        """
        Calculate memory used by model parameters in megabytes.

        ```
        Parameter Memory Formula:
        Memory (MB) = parameter_count × 4 bytes / (1024 × 1024)
                           ↑              ↑
                     From count_parameters  FP32 size
        ```

        Args:
            model: Model to analyze

        Returns:
            float: Parameter memory in megabytes
        """
        ### BEGIN SOLUTION
        param_count = self.count_parameters(model)
        return (param_count * BYTES_PER_FLOAT32) / MB_TO_BYTES
        ### END SOLUTION

    def _calculate_memory_efficiency(self, useful_memory_mb: float, peak_memory_mb: float) -> float:
        """
        Calculate memory efficiency as ratio of useful to total memory.

        ```
        Efficiency = useful_memory / peak_memory
                         ↑               ↑
              Parameters + Activations   tracemalloc peak

        Ideal: 1.0 (all memory is useful)
        Typical: 0.3-0.8 (overhead from allocator, fragmentation)
        ```

        Args:
            useful_memory_mb: Sum of parameter + activation memory
            peak_memory_mb: Peak memory observed by tracemalloc

        Returns:
            float: Efficiency ratio clamped to [0, 1]
        """
        ### BEGIN SOLUTION
        ratio = useful_memory_mb / max(peak_memory_mb, 0.001)
        return min(ratio, 1.0)
        ### END SOLUTION

    def measure_memory(self, model, input_shape: Tuple[int, ...]) -> Dict[str, float]:
        """
        Measure memory usage during forward pass.

        TODO: Implement memory tracking using tracemalloc and helper methods

        APPROACH:
        1. Use _calculate_parameter_memory() for parameter bytes
        2. Use tracemalloc to track peak allocation during forward pass
        3. Use _calculate_memory_efficiency() for efficiency ratio

        EXAMPLE:
        >>> linear = Linear(1024, 512)
        >>> profiler = Profiler()
        >>> memory = profiler.measure_memory(linear, (32, 1024))
        >>> print(f"Parameters: {memory['parameter_memory_mb']:.1f} MB")
        Parameters: 2.1 MB

        HINT: tracemalloc.start() / get_traced_memory() / stop() lifecycle
        """
        ### BEGIN SOLUTION
        tracemalloc.start()
        _baseline_memory = tracemalloc.get_traced_memory()[0]

        parameter_memory_mb = self._calculate_parameter_memory(model)

        dummy_input = Tensor(np.random.randn(*input_shape))
        activation_memory_mb = (dummy_input.data.nbytes * 2) / MB_TO_BYTES

        _ = model.forward(dummy_input)

        _current_memory, peak_memory = tracemalloc.get_traced_memory()
        peak_memory_mb = (peak_memory - _baseline_memory) / MB_TO_BYTES
        tracemalloc.stop()

        useful_memory = parameter_memory_mb + activation_memory_mb
        return {
            'parameter_memory_mb': parameter_memory_mb,
            'activation_memory_mb': activation_memory_mb,
            'peak_memory_mb': max(peak_memory_mb, useful_memory),
            'memory_efficiency': self._calculate_memory_efficiency(useful_memory, peak_memory_mb)
        }
        ### END SOLUTION

    def measure_latency(self, model, input_tensor, warmup: int = 10, iterations: int = 100) -> float:
        """
        Measure model inference latency with statistical rigor.

        TODO: Implement accurate latency measurement

        APPROACH:
        1. Run warmup iterations to stabilize performance
        2. Measure multiple iterations for statistical accuracy
        3. Calculate median latency to handle outliers
        4. Return latency in milliseconds

        PARAMETERS:
        - warmup: Number of warmup runs (default 10)
        - iterations: Number of measurement runs (default 100)

        EXAMPLE:
        >>> linear = Linear(128, 64)
        >>> input_tensor = Tensor(np.random.randn(1, 128))
        >>> profiler = Profiler()
        >>> latency = profiler.measure_latency(linear, input_tensor)
        >>> print(f"Latency: {latency:.2f} ms")
        Latency: 0.15 ms

        HINTS:
        - Use time.perf_counter() for high precision
        - Use median instead of mean for robustness against outliers
        - Handle different model interfaces (forward, __call__)
        """
        ### BEGIN SOLUTION
        # Warmup runs to stabilize performance
        for _ in range(warmup):
            _ = model.forward(input_tensor)

        # measurement runs
        times = []
        for _ in range(iterations):
            start_time = time.perf_counter()
            _ = model.forward(input_tensor)
            end_time = time.perf_counter()
            times.append((end_time - start_time) * 1000) # convert to milliseconds

        # calculate statistics - use median for robustness
        times = np.array(times)
        median_latency = np.median(times)

        return float(median_latency)
        ### END SOLUTION

    def profile_layer(self, layer, input_shape: Tuple[int, ...]) -> Dict[str, Any]:
        """
        Profile a single layer comprehensively.

        TODO: Implement layer-wise profiling

        APPROACH:
        1. Count parameters for this layer
        2. Count FLOPs for this layer
        3. Measure memory usage
        4. Measure latency
        5. Return comprehensive layer profile

        EXAMPLE:
        >>> linear = Linear(256, 128)
        >>> profiler = Profiler()
        >>> profile = profiler.profile_layer(linear, (32, 256))
        >>> print(f"Layer uses {profile['parameters']} parameters")
        Layer uses 32896 parameters

        HINTS:
        - Use existing profiler methods (count_parameters, count_flops, etc.)
        - Create dummy input for latency measurement
        - Include layer type information in profile
        """
        ### BEGIN SOLUTION
        # Create dummy input for latency measurement
        dummy_input = Tensor(np.random.randn(*input_shape))

        # gather all measurements
        params = self.count_parameters(layer)
        flops = self.count_flops(layer, input_shape)
        memory = self.measure_memory(layer, input_shape)
        latency = self.memory_latency(layer, dummy_input, warmup=3, iteractions=10)

        # compute derived metrics
        gflops_per_second = (flops / 1e9) / max(latency / 1000, 1e-6)

        return {
            'layer_type': layer.__class__.__name__,
            'parameters': params,
            'flops': flops,
            'latency_ms': latency,
            'gflops_per_second': gflops_per_second,
            **memory
        }
        ### END SOLUTION

    def _compute_derived_metrics(self, flops: int, latency_ms: float,
                                  peak_memory_mb: float) -> Dict[str, float]:
        """
        Compute throughput and efficiency metrics from raw measurements.

        ```
        Derived Metrics Pipeline:
        FLOPs + Latency → GFLOP/s (throughput)
        Memory + Latency → MB/s (bandwidth)
        GFLOP/s / Peak → Efficiency (utilization)
        ```

        Args:
            flops: Total floating point operations
            latency_ms: Measured latency in milliseconds
            peak_memory_mb: Peak memory usage in megabytes

        Returns:
            dict with gflops_per_second, memory_bandwidth_mbs, computational_efficiency
        """
        ### BEGIN SOLUTION
        latency_seconds = latency_ms / 1000.0
        gflops_per_second = (flops / 1e9) / max(latency_seconds, 1e-6)
        memory_bandwidth = peak_memory_mb / max(latency_seconds, 1e-6)
        theoretical_peak_gflops = 100.0
        computational_efficiency = min(gflops_per_second / theoretical_peak_gflops, 1.0)

        return {
            'gflops_per_second': gflops_per_second,
            'memory_bandwidth_mbs': memory_bandwidth,
            'computational_efficiency': computational_efficiency
        }
        ### END SOLUTION

    def _analyze_bottleneck(self, gflops_per_second: float,
                            memory_bandwidth_mbs: float) -> Dict[str, Any]:
        """
        Identify whether workload is memory-bound or compute-bound.

        ```
        Bottleneck Decision:
        If bandwidth >> GFLOP/s × 100 → Memory-bound (data movement dominates)
        Otherwise                      → Compute-bound (arithmetic dominates)
        ```

        Args:
            gflops_per_second: Compute throughput
            memory_bandwidth_mbs: Memory bandwidth in MB/s

        Returns:
            dict with is_memory_bound, is_compute_bound, bottleneck label
        """
        ### BEGIN SOLUTION
        is_memory_bound = memory_bandwidth_mbs > gflops_per_second * 100
        return {
            'is_memory_bound': is_memory_bound,
            'is_compute_bound': not is_memory_bound,
            'bottleneck': 'memory' if is_memory_bound else 'compute'
        }
        ### END SOLUTION

    def profile_forward_pass(self, model, input_tensor) -> Dict[str, Any]:
        """
        Comprehensive profiling of a model's forward pass.

        TODO: Gather measurements, then use _compute_derived_metrics and _analyze_bottleneck

        APPROACH:
        1. Gather raw measurements (parameters, FLOPs, memory, latency)
        2. Use _compute_derived_metrics() for throughput and efficiency
        3. Use _analyze_bottleneck() for bottleneck identification

        EXAMPLE:
        >>> model = Linear(256, 128)
        >>> input_data = Tensor(np.random.randn(32, 256))
        >>> profiler = Profiler()
        >>> profile = profiler.profile_forward_pass(model, input_data)
        >>> print(f"Throughput: {profile['gflops_per_second']:.2f} GFLOP/s")
        Throughput: 2.45 GFLOP/s

        HINT: Compose helper outputs with ** unpacking into return dict
        """
        ### BEGIN SOLUTION
        param_count = self.count_parameters(model)
        flops = self.count_flops(model, input_tensor.shape)
        memory_stats = self.measure_memory(model, input_tensor.shape)
        latency_ms = self.measure_latency(model, input_tensor, warmup=5, iterations=20)

        derived = self._compute_derived_metrics(flops, latency_ms, memory_stats['peak_memory_mb'])
        bottleneck = self._analyze_bottleneck(derived['gflops_per_second'],
                                              derived['memory_bandwidth_mbs'])
        
        return {
            'parameters': param_count, 'flops': flops, 'latency_ms': latency_ms,
            **memory_stats, **derived, **bottleneck
        }
        ### END SOLUTION

    def _estimate_backward_costs(self, forward_flops: int,
                                  forward_latency_ms: float) -> Dict[str, float]:
        """
        Estimate backward pass compute costs from forward pass measurements.

        ```
        Backward Pass Cost Estimation:
        Backward FLOPs   = Forward FLOPs × 2   (gradient computation)
        Backward Latency = Forward Latency × 2 (more complex operations)

        Why 2×? Each operation needs:
        1. Gradient w.r.t. weights (same cost as forward)
        2. Gradient w.r.t. inputs (same cost as forward)
        ```

        Args:
            forward_flops: FLOP count from forward pass
            forward_latency_ms: Latency from forward pass

        Returns:
            dict with backward_flops and backward_latency_ms
        """
        ### BEGIN SOLUTION
        return {
            'backward_flops': forward_flops * 2,
            'backward_latency_ms': forward_latency_ms * 2
        }
        ### END SOLUTION

    def _estimate_optimizer_memory(self, gradient_memory_mb: float) -> Dict[str, float]:
        """
        Estimate additional memory required by different optimizers.

        ```
        Optimizer Memory Requirements:
        ┌───────────┬────────────────────────────────────┐
        │ Optimizer │ Extra Memory                       │
        ├───────────┼────────────────────────────────────┤
        │ SGD       │ 0× (no state)                      │
        │ Adam      │ 2× gradient memory (m + v)         │
        │ AdamW     │ 2× gradient memory (m + v)         │
        └───────────┴────────────────────────────────────┘
        ```

        Args:
            gradient_memory_mb: Memory for gradient storage in MB

        Returns:
            dict mapping optimizer name to extra memory in MB
        """
        ### BEGIN SOLUTION
        return {
            'sgd': 0,
            'adam': gradient_memory_mb * 2,
            'adamw': gradient_memory_mb * 2
        }
        ### END SOLUTION

    def profile_backward_pass(self, model, input_tensor, _loss_fn=None) -> Dict[str, Any]:
        """
        Profile both forward and backward passes for training analysis.

        TODO: Use _estimate_backward_costs and _estimate_optimizer_memory helpers

        APPROACH:
        1. Profile forward pass with profile_forward_pass()
        2. Use _estimate_backward_costs() for backward FLOPs and latency
        3. Use _estimate_optimizer_memory() for optimizer memory estimates
        4. Combine into total training iteration metrics

        EXAMPLE:
        >>> model = Linear(128, 64)
        >>> input_data = Tensor(np.random.randn(16, 128))
        >>> profiler = Profiler()
        >>> profile = profiler.profile_backward_pass(model, input_data)
        >>> print(f"Training iteration: {profile['total_latency_ms']:.2f} ms")
        Training iteration: 0.45 ms

        HINT: Gradient memory equals parameter memory (one gradient per parameter)
        """
        ### BEGIN SOLUTION
        fwd = self.profile_forward_pass(model, input_tensor)
        bwd = self._estimate_backward_costs(fwd['flops'], fwd['latency_ms'])

        gradient_memory_mb = fwd['parameter_memory_mb']
        total_flops = fwd['flops'] + bwd['backward_flops']
        total_latency_ms = fwd['latency_ms'] + bwd['backward_latency_ms']
        total_memory_mb = fwd['parameter_memory_mb'] + fwd['activation_memory_mb'] + gradient_memory_mb

        return {
            'forward_flops': fwd['flops'],
            'forward_latency_ms': fwd['latency_ms'],
            'forward_memory_mb': fwd['peak_memory_mb'],
            **bwd,
            'gradient_memory_mb': gradient_memory_mb,
            'total_flops': total_flops,
            'total_latency_ms': total_latency_ms,
            'total_memory_mb': total_memory_mb,
            'total_gflops_per_second': (total_flops / 1e9) / (total_latency_ms / 1000.0),
            'optimizer_memory_estimates': self._estimate_optimizer_memory(gradient_memory_mb),
            'memory_efficiency': fwd['memory_efficiency'],
            'bottleneck': fwd['bottleneck']
        }
        ### END SOLUTION

In [21]:
#| export
def quick_profile(model, input_tensor, profiler=None):
    """
    Quick profiling function for immediate insights.

    Provides a simplified interface for profiling that displays key metrics
    in a student-friendly format.

    Args:
        model: Model to profile
        input_tensor: Input data for profiling
        profiler: Optional Profiler instance (creates new one if None)

    Returns:
        dict: Profile results with key metrics

    Example:
        >>> model = Linear(128, 64)
        >>> input_data = Tensor(np.random.randn(16, 128))
        >>> results = quick_profile(model, input_data)
        >>> # Displays formatted output automatically
    """
    if profiler is None:
        profiler = Profiler()

    profile = profiler.profile_forward_pass(model, input_tensor)

    # Display formatted results
    print("🧪 Quick Profile Results:")
    print(f"   Parameters: {profile['parameters']:,}")
    print(f"   FLOPs: {profile['flops']:,}")
    print(f"   Latency: {profile['latency_ms']:.2f} ms")
    print(f"   Memory: {profile['peak_memory_mb']:.2f} MB")
    print(f"   Bottleneck: {profile['bottleneck']}")
    print(f"   Efficiency: {profile['computational_efficiency']*100:.1f}%")

    return profile

In [22]:
#| export
def analyze_weight_distribution(model, percentiles=[10, 25, 50, 75, 90]):
    """
    Analyze weight distribution across layers.

    Helps understand how weights are distributed across layers.
    Useful for identifying patterns in parameter magnitudes.

    Args:
        model: Model to analyze
        percentiles: List of percentiles to compute

    Returns:
        dict: Weight distribution statistics

    Example:
        >>> model = Linear(512, 512)
        >>> stats = analyze_weight_distribution(model)
        >>> print(f"Weights < 0.01: {stats['below_threshold_001']:.1f}%")
    """
    # Collect all weights
    weights = []
    if hasattr(model, 'parameters'):
        for param in model.parameters():
            weights.extend(param.data.flatten().tolist())
    elif hasattr(model, 'weight'):
        weights.extend(model.weight.data.flatten().tolist())
    else:
        return {'error': 'No weights found'}

    weights = np.array(weights)
    abs_weights = np.abs(weights)

    # Calculate statistics
    stats = {
        'total_weights': len(weights),
        'mean': float(np.mean(abs_weights)),
        'std': float(np.std(abs_weights)),
        'min': float(np.min(abs_weights)),
        'max': float(np.max(abs_weights)),
    }

    # Percentile analysis
    for p in percentiles:
        stats[f'percentile_{p}'] = float(np.percentile(abs_weights, p))

    # Threshold analysis (useful for pruning)
    for threshold in [0.001, 0.01, 0.1]:
        below = np.sum(abs_weights < threshold) / len(weights) * 100
        stats[f'below_threshold_{str(threshold).replace(".", "")}'] = below

    return stats

In [23]:
def test_unit_helper_functions():
    """🧪 Test helper function implementations."""
    print("🧪 Unit Test: Helper Functions...")

    # Test 1: Quick profile function
    from tinytorch.core.layers import Linear
    test_model = Linear(16, 8)
    test_input = Tensor(np.random.randn(8, 16))
    profile = quick_profile(test_model, test_input, profiler=Profiler())

    # Validate profile contains expected keys
    assert 'parameters' in profile, "Quick profile should include parameters"
    assert 'flops' in profile, "Quick profile should include FLOPs"
    assert 'latency_ms' in profile, "Quick profile should include latency"
    print("✅ Quick profile provides comprehensive metrics")

    # Test 2: Weight distribution analysis
    class SimpleModel:
        def __init__(self):
            self.weight = Tensor(np.random.randn(10, 5) * 0.1)  # Small weights

    model = SimpleModel()
    stats = analyze_weight_distribution(model)

    # Validate statistics structure
    assert 'total_weights' in stats, "Should count total weights"
    assert 'mean' in stats, "Should compute mean"
    assert 'std' in stats, "Should compute standard deviation"
    assert stats['total_weights'] == 50, f"Expected 50 weights, got {stats['total_weights']}"
    print(f"✅ Weight distribution analysis: {stats['total_weights']} weights analyzed")

    # Test 3: Weight distribution with no weights
    class NoWeightModel:
        pass

    no_weight_model = NoWeightModel()
    stats = analyze_weight_distribution(no_weight_model)
    assert 'error' in stats, "Should handle models without weights"
    print("✅ Handles models without weights gracefully")

    print("✅ Helper functions work correctly!")

if __name__ == "__main__":
    test_unit_helper_functions()

🧪 Unit Test: Helper Functions...
🧪 Quick Profile Results:
   Parameters: 136
   FLOPs: 256
   Latency: 0.15 ms
   Memory: 0.01 MB
   Bottleneck: memory
   Efficiency: 0.0%
✅ Quick profile provides comprehensive metrics
✅ Weight distribution analysis: 50 weights analyzed
✅ Handles models without weights gracefully
✅ Helper functions work correctly!


In [24]:
def test_unit_count_layer_parameters():
    """🧪 Test _count_layer_parameters helper."""
    print("🧪 Unit Test: _count_layer_parameters...")

    profiler = Profiler()

    # Test 1: Layer with weight and bias
    class LayerWithBias:
        def __init__(self):
            self.weight = Tensor(np.random.randn(10, 5))
            self.bias = Tensor(np.random.randn(5))

    layer = LayerWithBias()
    count = profiler._count_layer_parameters(layer)
    assert count == 55, f"Expected 55 (10*5 + 5), got {count}"
    print(f"✅ Layer with bias: {count} parameters")

    # Test 2: Layer with weight only (no bias)
    class LayerNoBias:
        def __init__(self):
            self.weight = Tensor(np.random.randn(8, 4))

    layer_no_bias = LayerNoBias()
    count = profiler._count_layer_parameters(layer_no_bias)
    assert count == 32, f"Expected 32 (8*4), got {count}"
    print(f"✅ Layer without bias: {count} parameters")

    # Test 3: Object without weight attribute
    class NoWeight:
        pass

    count = profiler._count_layer_parameters(NoWeight())
    assert count == 0, f"Expected 0, got {count}"
    print("✅ No weight attribute: 0 parameters")

    print("✅ _count_layer_parameters works correctly!")

if __name__ == "__main__":
    test_unit_count_layer_parameters()

🧪 Unit Test: _count_layer_parameters...
✅ Layer with bias: 55 parameters
✅ Layer without bias: 32 parameters
✅ No weight attribute: 0 parameters
✅ _count_layer_parameters works correctly!


In [25]:
def test_unit_parameter_counting():
    """🧪 Test parameter counting implementation."""
    print("🧪 Unit Test: Parameter Counting...")

    profiler = Profiler()

    # Test 1: Simple model with known parameters
    class SimpleModel:
        def __init__(self):
            self.weight = Tensor(np.random.randn(10, 5))
            self.bias = Tensor(np.random.randn(5))

        def parameters(self):
            return [self.weight, self.bias]

    simple_model = SimpleModel()
    param_count = profiler.count_parameters(simple_model)
    expected_count = 10 * 5 + 5  # weight + bias
    assert param_count == expected_count, f"Expected {expected_count} parameters, got {param_count}"
    print(f"✅ Simple model: {param_count} parameters")

    # Test 2: Model without parameters
    class NoParamModel:
        def __init__(self):
            pass

    no_param_model = NoParamModel()
    param_count = profiler.count_parameters(no_param_model)
    assert param_count == 0, f"Expected 0 parameters, got {param_count}"
    print(f"✅ No parameter model: {param_count} parameters")

    # Test 3: Direct tensor (no parameters)
    test_tensor = Tensor(np.random.randn(2, 3))
    param_count = profiler.count_parameters(test_tensor)
    assert param_count == 0, f"Expected 0 parameters for tensor, got {param_count}"
    print(f"✅ Direct tensor: {param_count} parameters")

    print("✅ Parameter counting works correctly!")

if __name__ == "__main__":
    test_unit_parameter_counting()

🧪 Unit Test: Parameter Counting...
✅ Simple model: 55 parameters
✅ No parameter model: 0 parameters
✅ Direct tensor: 0 parameters
✅ Parameter counting works correctly!


In [26]:
def test_unit_count_linear_flops():
    """🧪 Test _count_linear_flops helper."""
    print("🧪 Unit Test: _count_linear_flops...")

    profiler = Profiler()

    # Create mock Linear layer
    class MockLinear:
        def __init__(self, in_f, out_f):
            self.weight = Tensor(np.random.randn(in_f, out_f))
            self.__class__.__name__ = 'Linear'

    # Test 1: Known dimensions
    layer = MockLinear(128, 64)
    flops = profiler._count_linear_flops(layer, (1, 128))
    assert flops == 128 * 64 * 2, f"Expected {128*64*2}, got {flops}"
    print(f"✅ Linear(128, 64): {flops} FLOPs")

    # Test 2: Square layer
    layer_sq = MockLinear(256, 256)
    flops_sq = profiler._count_linear_flops(layer_sq, (1, 256))
    assert flops_sq == 256 * 256 * 2, f"Expected {256*256*2}, got {flops_sq}"
    print(f"✅ Linear(256, 256): {flops_sq} FLOPs")

    # Test 3: Batch independence (uses last dim only)
    flops_b1 = profiler._count_linear_flops(layer, (1, 128))
    flops_b32 = profiler._count_linear_flops(layer, (32, 128))
    assert flops_b1 == flops_b32, "FLOPs should be batch-independent"
    print("✅ Batch-independent FLOPs confirmed")

    print("✅ _count_linear_flops works correctly!")

if __name__ == "__main__":
    test_unit_count_linear_flops()

🧪 Unit Test: _count_linear_flops...
✅ Linear(128, 64): 16384 FLOPs
✅ Linear(256, 256): 131072 FLOPs
✅ Batch-independent FLOPs confirmed
✅ _count_linear_flops works correctly!


In [27]:
def test_unit_count_conv_flops():
    """🧪 Test _count_conv_flops helper."""
    print("🧪 Unit Test: _count_conv_flops...")

    profiler = Profiler()

    # Create mock Conv2d layer
    class MockConv:
        def __init__(self, in_c, out_c, k, s=1):
            self.in_channels = in_c
            self.out_channels = out_c
            self.kernel_size = k
            self.stride = s
            self.__class__.__name__ = 'Conv2d'

    # Test 1: Simple 3x3 conv, stride 1
    conv = MockConv(3, 16, 3, 1)
    flops = profiler._count_conv_flops(conv, (1, 3, 32, 32))
    expected = 32 * 32 * 3 * 3 * 3 * 16 * 2
    assert flops == expected, f"Expected {expected}, got {flops}"
    print(f"✅ Conv2d(3, 16, 3): {flops} FLOPs")

    # Test 2: Stride 2 halves output spatial dims
    conv_s2 = MockConv(3, 64, 7, 2)
    flops_s2 = profiler._count_conv_flops(conv_s2, (1, 3, 224, 224))
    out_h, out_w = 224 // 2, 224 // 2
    expected_s2 = out_h * out_w * 7 * 7 * 3 * 64 * 2
    assert flops_s2 == expected_s2, f"Expected {expected_s2}, got {flops_s2}"
    print(f"✅ Conv2d(3, 64, 7, stride=2): {flops_s2} FLOPs")

    # Test 3: Missing attributes returns 0
    class Incomplete:
        pass

    assert profiler._count_conv_flops(Incomplete(), (1, 3, 32, 32)) == 0
    print("✅ Missing attributes returns 0")

    print("✅ _count_conv_flops works correctly!")

if __name__ == "__main__":
    test_unit_count_conv_flops()

🧪 Unit Test: _count_conv_flops...
✅ Conv2d(3, 16, 3): 884736 FLOPs
✅ Conv2d(3, 64, 7, stride=2): 236027904 FLOPs
✅ Missing attributes returns 0
✅ _count_conv_flops works correctly!


In [28]:
def test_unit_count_sequential_flops():
    """🧪 Test _count_sequential_flops helper."""
    print("🧪 Unit Test: _count_sequential_flops...")

    profiler = Profiler()

    # Create mock sequential model with two Linear layers
    class MockLinear:
        def __init__(self, in_f, out_f):
            self.weight = Tensor(np.random.randn(in_f, out_f))
            self.__class__.__name__ = 'Linear'

    class MockSequential:
        def __init__(self, *layer_list):
            self.layers = list(layer_list)

    model = MockSequential(MockLinear(128, 64), MockLinear(64, 10))
    total_flops = profiler._count_sequential_flops(model, (1, 128))

    expected = (128 * 64 * 2) + (64 * 10 * 2)
    assert total_flops == expected, f"Expected {expected}, got {total_flops}"
    print(f"✅ Sequential(128->64->10): {total_flops} FLOPs")

    # Single layer sequential
    model_single = MockSequential(MockLinear(32, 16))
    flops_single = profiler._count_sequential_flops(model_single, (1, 32))
    assert flops_single == 32 * 16 * 2, f"Expected {32*16*2}, got {flops_single}"
    print(f"✅ Single-layer sequential: {flops_single} FLOPs")

    print("✅ _count_sequential_flops works correctly!")

if __name__ == "__main__":
    test_unit_count_sequential_flops()

🧪 Unit Test: _count_sequential_flops...
✅ Sequential(128->64->10): 17664 FLOPs
✅ Single-layer sequential: 1024 FLOPs
✅ _count_sequential_flops works correctly!


In [29]:
def test_unit_flop_counting():
    """🧪 Test FLOP counting implementation."""
    print("🧪 Unit Test: FLOP Counting...")

    profiler = Profiler()

    # Test 1: Simple tensor operations
    test_tensor = Tensor(np.random.randn(4, 8))
    flops = profiler.count_flops(test_tensor, (4, 8))
    expected_flops = 4 * 8  # 1 FLOP per element for generic operation
    assert flops == expected_flops, f"Expected {expected_flops} FLOPs, got {flops}"
    print(f"✅ Tensor operation: {flops} FLOPs")

    # Test 2: Simulated Linear layer
    class MockLinear:
        def __init__(self, in_features, out_features):
            self.weight = Tensor(np.random.randn(in_features, out_features))
            self.__class__.__name__ = 'Linear'

    mock_linear = MockLinear(128, 64)
    flops = profiler.count_flops(mock_linear, (1, 128))
    expected_flops = 128 * 64 * 2  # matmul FLOPs
    assert flops == expected_flops, f"Expected {expected_flops} FLOPs, got {flops}"
    print(f"✅ Linear layer: {flops} FLOPs")

    # Test 3: Batch size independence
    flops_batch1 = profiler.count_flops(mock_linear, (1, 128))
    flops_batch32 = profiler.count_flops(mock_linear, (32, 128))
    assert flops_batch1 == flops_batch32, "FLOPs should be independent of batch size"
    print(f"✅ Batch independence: {flops_batch1} FLOPs (same for batch 1 and 32)")

    print("✅ FLOP counting works correctly!")

if __name__ == "__main__":
    test_unit_flop_counting()

🧪 Unit Test: FLOP Counting...
✅ Tensor operation: 32 FLOPs
✅ Linear layer: 16384 FLOPs
✅ Batch independence: 16384 FLOPs (same for batch 1 and 32)
✅ FLOP counting works correctly!


In [30]:
def test_unit_calculate_parameter_memory():
    """🧪 Test _calculate_parameter_memory helper."""
    print("🧪 Unit Test: _calculate_parameter_memory...")

    profiler = Profiler()

    # Test 1: Known parameter count -> known memory
    class KnownModel:
        def __init__(self):
            # 1024 * 1024 = 1,048,576 parameters = exactly 4 MB at FP32
            self.weight = Tensor(np.random.randn(1024, 1024))

    model = KnownModel()
    memory_mb = profiler._calculate_parameter_memory(model)
    expected_mb = (1024 * 1024 * 4) / (1024 * 1024)  # 4.0 MB
    assert abs(memory_mb - expected_mb) < 0.01, f"Expected {expected_mb} MB, got {memory_mb}"
    print(f"✅ 1M params = {memory_mb:.1f} MB")

    # Test 2: Zero parameter model
    class EmptyModel:
        pass

    empty_mb = profiler._calculate_parameter_memory(EmptyModel())
    assert empty_mb == 0.0, f"Expected 0.0 MB, got {empty_mb}"
    print("✅ Empty model = 0.0 MB")

    print("✅ _calculate_parameter_memory works correctly!")

if __name__ == "__main__":
    test_unit_calculate_parameter_memory()

🧪 Unit Test: _calculate_parameter_memory...
✅ 1M params = 4.0 MB
✅ Empty model = 0.0 MB
✅ _calculate_parameter_memory works correctly!


In [31]:
def test_unit_calculate_memory_efficiency():
    """🧪 Test _calculate_memory_efficiency helper."""
    print("🧪 Unit Test: _calculate_memory_efficiency...")

    profiler = Profiler()

    # Test 1: Perfect efficiency
    eff = profiler._calculate_memory_efficiency(10.0, 10.0)
    assert abs(eff - 1.0) < 0.01, f"Expected 1.0, got {eff}"
    print(f"✅ Perfect efficiency: {eff}")

    # Test 2: Half efficiency
    eff_half = profiler._calculate_memory_efficiency(5.0, 10.0)
    assert abs(eff_half - 0.5) < 0.01, f"Expected 0.5, got {eff_half}"
    print(f"✅ Half efficiency: {eff_half}")

    # Test 3: Clamped at 1.0 (useful > peak shouldn't exceed 1.0)
    eff_clamped = profiler._calculate_memory_efficiency(20.0, 10.0)
    assert eff_clamped <= 1.0, f"Efficiency should be clamped to 1.0, got {eff_clamped}"
    print(f"✅ Clamped efficiency: {eff_clamped}")

    # Test 4: Division by zero safety
    eff_zero = profiler._calculate_memory_efficiency(5.0, 0.0)
    assert eff_zero <= 1.0, f"Should handle zero peak safely, got {eff_zero}"
    print("✅ Zero-peak safety handled")

    print("✅ _calculate_memory_efficiency works correctly!")

if __name__ == "__main__":
    test_unit_calculate_memory_efficiency()

🧪 Unit Test: _calculate_memory_efficiency...
✅ Perfect efficiency: 1.0
✅ Half efficiency: 0.5
✅ Clamped efficiency: 1.0
✅ Zero-peak safety handled
✅ _calculate_memory_efficiency works correctly!


In [32]:
def test_unit_memory_measurement():
    """🧪 Test memory measurement implementation."""
    print("🧪 Unit Test: Memory Measurement...")

    profiler = Profiler()

    # Test 1: Basic memory measurement
    test_tensor = Tensor(np.random.randn(10, 20))
    from tinytorch.core.layers import Linear
    test_model = Linear(20, 10)
    memory_stats = profiler.measure_memory(test_model, (10, 20))

    # Validate dictionary structure
    required_keys = ['parameter_memory_mb', 'activation_memory_mb', 'peak_memory_mb', 'memory_efficiency']
    for key in required_keys:
        assert key in memory_stats, f"Missing key: {key}"

    # Validate non-negative values
    for key in required_keys:
        assert memory_stats[key] >= 0, f"{key} should be non-negative, got {memory_stats[key]}"

    print(f"✅ Basic measurement: {memory_stats['peak_memory_mb']:.3f} MB peak")

    # Test 2: Memory scaling with size
    from tinytorch.core.layers import Linear
    small_model = Linear(5, 5)
    large_model = Linear(50, 50)

    small_memory = profiler.measure_memory(small_model, (5, 5))
    large_memory = profiler.measure_memory(large_model, (50, 50))

    # Larger tensor should use more activation memory
    assert large_memory['activation_memory_mb'] >= small_memory['activation_memory_mb'], \
        "Larger tensor should use more activation memory"

    print(f"✅ Scaling: Small {small_memory['activation_memory_mb']:.3f} MB → Large {large_memory['activation_memory_mb']:.3f} MB")

    # Test 3: Efficiency bounds
    assert 0 <= memory_stats['memory_efficiency'] <= 1.0, \
        f"Memory efficiency should be between 0 and 1, got {memory_stats['memory_efficiency']}"

    print(f"✅ Efficiency: {memory_stats['memory_efficiency']:.3f} (0-1 range)")

    print("✅ Memory measurement works correctly!")

if __name__ == "__main__":
    test_unit_memory_measurement()

🧪 Unit Test: Memory Measurement...
✅ Basic measurement: 0.003 MB peak
✅ Scaling: Small 0.000 MB → Large 0.019 MB
✅ Efficiency: 0.669 (0-1 range)
✅ Memory measurement works correctly!


In [33]:
def test_unit_latency_measurement():
    """🧪 Test latency measurement implementation."""
    print("🧪 Unit Test: Latency Measurement...")

    profiler = Profiler()

    # Test 1: Basic latency measurement
    from tinytorch.core.layers import Linear
    test_model = Linear(8, 4)
    test_input = Tensor(np.random.randn(4, 8))
    latency = profiler.measure_latency(test_model, test_input, warmup=2, iterations=5)

    assert latency >= 0, f"Latency should be non-negative, got {latency}"
    assert latency < 1000, f"Latency seems too high for simple operation: {latency} ms"
    print(f"✅ Basic latency: {latency:.3f} ms")

    # Test 2: Measurement consistency
    latencies = []
    for _ in range(3):
        lat = profiler.measure_latency(test_model, test_input, warmup=1, iterations=3)
        latencies.append(lat)

    # Measurements should be in reasonable range
    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    assert std_latency < avg_latency, "Standard deviation shouldn't exceed mean for simple operations"
    print(f"✅ Consistency: {avg_latency:.3f} ± {std_latency:.3f} ms")

    # Test 3: Size scaling
    small_model = Linear(2, 2)
    large_model = Linear(20, 20)
    small_input = Tensor(np.random.randn(2, 2))
    large_input = Tensor(np.random.randn(20, 20))

    small_latency = profiler.measure_latency(small_model, small_input, warmup=1, iterations=3)
    large_latency = profiler.measure_latency(large_model, large_input, warmup=1, iterations=3)

    # Larger operations might take longer (though not guaranteed for simple operations)
    print(f"✅ Scaling: Small {small_latency:.3f} ms, Large {large_latency:.3f} ms")

    print("✅ Latency measurement works correctly!")

if __name__ == "__main__":
    test_unit_latency_measurement()

🧪 Unit Test: Latency Measurement...
✅ Basic latency: 0.047 ms
✅ Consistency: 0.026 ± 0.017 ms
✅ Scaling: Small 0.006 ms, Large 0.232 ms
✅ Latency measurement works correctly!


In [34]:
def test_unit_compute_derived_metrics():
    """🧪 Test _compute_derived_metrics helper."""
    print("🧪 Unit Test: _compute_derived_metrics...")

    profiler = Profiler()

    # Test 1: Known values -> known throughput
    # 1e9 FLOPs in 1000ms (1 second) = 1.0 GFLOP/s
    metrics = profiler._compute_derived_metrics(
        flops=1_000_000_000, latency_ms=1000.0, peak_memory_mb=100.0
    )
    assert abs(metrics['gflops_per_second'] - 1.0) < 0.01, \
        f"Expected 1.0 GFLOP/s, got {metrics['gflops_per_second']}"
    print(f"✅ 1B FLOPs / 1s = {metrics['gflops_per_second']:.1f} GFLOP/s")

    # Test 2: Memory bandwidth calculation
    # 100 MB in 1 second = 100 MB/s
    assert abs(metrics['memory_bandwidth_mbs'] - 100.0) < 0.1, \
        f"Expected 100 MB/s, got {metrics['memory_bandwidth_mbs']}"
    print(f"✅ Memory bandwidth: {metrics['memory_bandwidth_mbs']:.1f} MB/s")

    # Test 3: Efficiency bounded by [0, 1]
    assert 0 <= metrics['computational_efficiency'] <= 1.0, \
        f"Efficiency out of bounds: {metrics['computational_efficiency']}"
    print(f"✅ Efficiency: {metrics['computational_efficiency']:.3f}")

    print("✅ _compute_derived_metrics works correctly!")

if __name__ == "__main__":
    test_unit_compute_derived_metrics()

🧪 Unit Test: _compute_derived_metrics...
✅ 1B FLOPs / 1s = 1.0 GFLOP/s
✅ Memory bandwidth: 100.0 MB/s
✅ Efficiency: 0.010
✅ _compute_derived_metrics works correctly!


In [35]:
def test_unit_analyze_bottleneck():
    """🧪 Test _analyze_bottleneck helper."""
    print("🧪 Unit Test: _analyze_bottleneck...")

    profiler = Profiler()

    # Test 1: Memory-bound (high bandwidth relative to compute)
    result = profiler._analyze_bottleneck(gflops_per_second=1.0, memory_bandwidth_mbs=10000.0)
    assert result['is_memory_bound'] is True, "High bandwidth should be memory-bound"
    assert result['bottleneck'] == 'memory'
    print("✅ High bandwidth -> memory-bound")

    # Test 2: Compute-bound (low bandwidth relative to compute)
    result = profiler._analyze_bottleneck(gflops_per_second=50.0, memory_bandwidth_mbs=100.0)
    assert result['is_compute_bound'] is True, "Low bandwidth should be compute-bound"
    assert result['bottleneck'] == 'compute'
    print("✅ Low bandwidth -> compute-bound")

    # Test 3: Mutually exclusive flags
    result = profiler._analyze_bottleneck(gflops_per_second=10.0, memory_bandwidth_mbs=500.0)
    assert result['is_memory_bound'] != result['is_compute_bound'], \
        "Memory-bound and compute-bound should be mutually exclusive"
    print(f"✅ Mutually exclusive: bottleneck = {result['bottleneck']}")

    print("✅ _analyze_bottleneck works correctly!")

if __name__ == "__main__":
    test_unit_analyze_bottleneck()

🧪 Unit Test: _analyze_bottleneck...
✅ High bandwidth -> memory-bound
✅ Low bandwidth -> compute-bound
✅ Mutually exclusive: bottleneck = compute
✅ _analyze_bottleneck works correctly!


In [36]:
def test_unit_estimate_backward_costs():
    """🧪 Test _estimate_backward_costs helper."""
    print("🧪 Unit Test: _estimate_backward_costs...")

    profiler = Profiler()

    # Test 1: Known forward values -> 2x backward
    costs = profiler._estimate_backward_costs(forward_flops=1000, forward_latency_ms=5.0)
    assert costs['backward_flops'] == 2000, f"Expected 2000, got {costs['backward_flops']}"
    assert costs['backward_latency_ms'] == 10.0, f"Expected 10.0, got {costs['backward_latency_ms']}"
    print(f"✅ 1000 forward FLOPs -> {costs['backward_flops']} backward FLOPs")

    # Test 2: Zero forward -> zero backward
    costs_zero = profiler._estimate_backward_costs(forward_flops=0, forward_latency_ms=0.0)
    assert costs_zero['backward_flops'] == 0
    assert costs_zero['backward_latency_ms'] == 0.0
    print("✅ Zero forward -> zero backward")

    print("✅ _estimate_backward_costs works correctly!")

if __name__ == "__main__":
    test_unit_estimate_backward_costs()

🧪 Unit Test: _estimate_backward_costs...
✅ 1000 forward FLOPs -> 2000 backward FLOPs
✅ Zero forward -> zero backward
✅ _estimate_backward_costs works correctly!


In [37]:
def test_unit_estimate_optimizer_memory():
    """🧪 Test _estimate_optimizer_memory helper."""
    print("🧪 Unit Test: _estimate_optimizer_memory...")

    profiler = Profiler()

    # Test with 100 MB gradient memory
    estimates = profiler._estimate_optimizer_memory(gradient_memory_mb=100.0)

    assert estimates['sgd'] == 0, f"SGD should need 0 extra, got {estimates['sgd']}"
    assert estimates['adam'] == 200.0, f"Adam should need 200 MB, got {estimates['adam']}"
    assert estimates['adamw'] == 200.0, f"AdamW should need 200 MB, got {estimates['adamw']}"
    print(f"✅ SGD: {estimates['sgd']} MB, Adam: {estimates['adam']} MB, AdamW: {estimates['adamw']} MB")

    # Test with zero gradients
    estimates_zero = profiler._estimate_optimizer_memory(gradient_memory_mb=0.0)
    assert estimates_zero['adam'] == 0.0, "Zero gradients -> zero optimizer memory"
    print("✅ Zero gradient memory handled correctly")

    print("✅ _estimate_optimizer_memory works correctly!")

if __name__ == "__main__":
    test_unit_estimate_optimizer_memory()

🧪 Unit Test: _estimate_optimizer_memory...
✅ SGD: 0 MB, Adam: 200.0 MB, AdamW: 200.0 MB
✅ Zero gradient memory handled correctly
✅ _estimate_optimizer_memory works correctly!


In [38]:
def test_unit_advanced_profiling():
    """🧪 Test advanced profiling functions."""
    print("🧪 Unit Test: Advanced Profiling Functions...")

    # Create profiler and test model
    profiler = Profiler()
    from tinytorch.core.layers import Linear
    test_model = Linear(8, 4)
    test_input = Tensor(np.random.randn(4, 8))

    # Test forward pass profiling
    forward_profile = profiler.profile_forward_pass(test_model, test_input)

    # Validate forward profile structure
    required_forward_keys = [
        'parameters', 'flops', 'latency_ms', 'gflops_per_second',
        'memory_bandwidth_mbs', 'bottleneck'
    ]

    for key in required_forward_keys:
        assert key in forward_profile, f"Missing key: {key}"

    assert forward_profile['parameters'] >= 0
    assert forward_profile['flops'] >= 0
    assert forward_profile['latency_ms'] >= 0
    assert forward_profile['gflops_per_second'] >= 0

    print(f"✅ Forward profiling: {forward_profile['gflops_per_second']:.2f} GFLOP/s")

    # Test backward pass profiling
    backward_profile = profiler.profile_backward_pass(test_model, test_input)

    # Validate backward profile structure
    required_backward_keys = [
        'forward_flops', 'backward_flops', 'total_flops',
        'total_latency_ms', 'total_memory_mb', 'optimizer_memory_estimates'
    ]

    for key in required_backward_keys:
        assert key in backward_profile, f"Missing key: {key}"

    # Validate relationships
    assert backward_profile['total_flops'] >= backward_profile['forward_flops']
    assert backward_profile['total_latency_ms'] >= backward_profile['forward_latency_ms']
    assert 'sgd' in backward_profile['optimizer_memory_estimates']
    assert 'adam' in backward_profile['optimizer_memory_estimates']

    # Check backward pass estimates are reasonable
    assert backward_profile['backward_flops'] >= backward_profile['forward_flops'], \
        "Backward pass should have at least as many FLOPs as forward"
    assert backward_profile['gradient_memory_mb'] >= 0, \
        "Gradient memory should be non-negative"

    print(f"✅ Backward profiling: {backward_profile['total_latency_ms']:.2f} ms total")
    print(f"✅ Memory breakdown: {backward_profile['total_memory_mb']:.2f} MB training")
    print("✅ Advanced profiling functions work correctly!")

if __name__ == "__main__":
    test_unit_advanced_profiling()

🧪 Unit Test: Advanced Profiling Functions...
✅ Forward profiling: 0.00 GFLOP/s
✅ Backward profiling: 0.04 ms total
✅ Memory breakdown: 0.00 MB training
✅ Advanced profiling functions work correctly!


In [39]:
def analyze_model_scaling():
    """📊 Analyze how model performance scales with size."""
    print("📊 Analyzing Model Scaling Characteristics...")

    profiler = Profiler()
    results = []

    # Test different model sizes
    sizes = [64, 128, 256, 512]

    print("\nModel Scaling Analysis:")
    print("Size\tParams\t\tFLOPs\t\tLatency(ms)\tMemory(MB)\tGFLOP/s")
    print("-" * 80)

    for size in sizes:
        # Create models of different sizes for comparison
        from tinytorch.core.layers import Linear
        test_model = Linear(size, size)
        input_shape = (32, size)  # Batch of 32
        dummy_input = Tensor(np.random.randn(*input_shape))

        # Simulate linear layer characteristics
        linear_params = size * size + size  # W + b
        linear_flops = size * size * 2  # matmul

        # Measure actual performance
        latency = profiler.measure_latency(test_model, dummy_input, warmup=3, iterations=10)
        memory = profiler.measure_memory(test_model, input_shape)

        gflops_per_second = (linear_flops / 1e9) / (latency / 1000)

        results.append({
            'size': size,
            'parameters': linear_params,
            'flops': linear_flops,
            'latency_ms': latency,
            'memory_mb': memory['peak_memory_mb'],
            'gflops_per_second': gflops_per_second
        })

        print(f"{size}\t{linear_params:,}\t\t{linear_flops:,}\t\t"
              f"{latency:.2f}\t\t{memory['peak_memory_mb']:.2f}\t\t"
              f"{gflops_per_second:.2f}")

    # Analysis insights
    print("\n💡 Scaling Analysis Insights:")

    # Memory scaling
    memory_growth = results[-1]['memory_mb'] / max(results[0]['memory_mb'], 0.001)
    print(f"Memory grows {memory_growth:.1f}× from {sizes[0]} to {sizes[-1]} size")

    # Compute scaling
    compute_growth = results[-1]['gflops_per_second'] / max(results[0]['gflops_per_second'], 0.001)
    print(f"Compute efficiency changes {compute_growth:.1f}× with size")

    # Performance characteristics
    avg_efficiency = np.mean([r['gflops_per_second'] for r in results])
    if avg_efficiency < 10:  # Arbitrary threshold for "low" efficiency
        print("🚀 Low compute efficiency suggests memory-bound workload")
    else:
        print("🚀 High compute efficiency suggests compute-bound workload")

def analyze_batch_size_effects():
    """📊 Analyze how batch size affects performance and efficiency."""
    print("\n📊 Analyzing Batch Size Effects...")

    profiler = Profiler()
    batch_sizes = [1, 8, 32, 128]
    feature_size = 256

    print("\nBatch Size Effects Analysis:")
    print("Batch\tLatency(ms)\tThroughput(samples/s)\tMemory(MB)\tMemory Efficiency")
    print("-" * 85)

    for batch_size in batch_sizes:
        from tinytorch.core.layers import Linear
        test_model = Linear(feature_size, feature_size)
        input_shape = (batch_size, feature_size)
        dummy_input = Tensor(np.random.randn(*input_shape))

        # Measure performance
        latency = profiler.measure_latency(test_model, dummy_input, warmup=3, iterations=10)
        memory = profiler.measure_memory(test_model, input_shape)

        # Calculate throughput
        samples_per_second = (batch_size * 1000) / latency  # samples/second

        # Calculate efficiency (samples per unit memory)
        efficiency = samples_per_second / max(memory['peak_memory_mb'], 0.001)

        print(f"{batch_size}\t{latency:.2f}\t\t{samples_per_second:.0f}\t\t\t"
              f"{memory['peak_memory_mb']:.2f}\t\t{efficiency:.1f}")

    print("\n💡 Batch Size Insights:")
    print("Larger batches typically improve throughput but increase memory usage")

# Run the analysis
if __name__ == "__main__":
    analyze_model_scaling()
    analyze_batch_size_effects()

📊 Analyzing Model Scaling Characteristics...

Model Scaling Analysis:
Size	Params		FLOPs		Latency(ms)	Memory(MB)	GFLOP/s
--------------------------------------------------------------------------------
64	4,160		8,192		1.16		0.03		0.01
128	16,512		32,768		2.37		0.09		0.01
256	65,792		131,072		5.86		0.31		0.02
512	262,656		524,288		14.50		1.13		0.04

💡 Scaling Analysis Insights:
Memory grows 34.3× from 64 to 512 size
Compute efficiency changes 5.1× with size
🚀 Low compute efficiency suggests memory-bound workload

📊 Analyzing Batch Size Effects...

Batch Size Effects Analysis:
Batch	Latency(ms)	Throughput(samples/s)	Memory(MB)	Memory Efficiency
-------------------------------------------------------------------------------------
1	0.18		5489			0.25		21703.5
8	1.47		5441			0.27		20410.3
32	6.04		5299			0.31		16902.9
128	25.68		4985			0.50		9950.1

💡 Batch Size Insights:
Larger batches typically improve throughput but increase memory usage


In [40]:
def benchmark_operation_efficiency():
    """📊 Compare efficiency of different operations for optimization guidance."""
    print("📊 Benchmarking Operation Efficiency...")

    profiler = Profiler()
    operations = []

    # Test different operation types
    size = 256
    input_tensor = Tensor(np.random.randn(32, size))

    # Elementwise operations (memory-bound)
    # Create a simple model wrapper for elementwise operations
    class ElementwiseModel:
        def forward(self, x):
            return x + x  # Simple elementwise operation

    elementwise_model = ElementwiseModel()
    elementwise_latency = profiler.measure_latency(elementwise_model, input_tensor, iterations=20)
    elementwise_flops = size * 32  # One operation per element

    operations.append({
        'operation': 'Elementwise',
        'latency_ms': elementwise_latency,
        'flops': elementwise_flops,
        'gflops_per_second': (elementwise_flops / 1e9) / (elementwise_latency / 1000),
        'efficiency_class': 'memory-bound',
        'optimization_focus': 'data_locality'
    })

    # Matrix operations (compute-bound)
    from tinytorch.core.layers import Linear
    matrix_model = Linear(size, size)
    matrix_latency = profiler.measure_latency(matrix_model, input_tensor, iterations=10)
    matrix_flops = size * size * 2  # Matrix multiplication

    operations.append({
        'operation': 'Matrix Multiply',
        'latency_ms': matrix_latency,
        'flops': matrix_flops,
        'gflops_per_second': (matrix_flops / 1e9) / (matrix_latency / 1000),
        'efficiency_class': 'compute-bound',
        'optimization_focus': 'algorithms'
    })

    # Reduction operations (memory-bound)
    class ReductionModel:
        def forward(self, x):
            return x.sum()  # Sum reduction operation

    reduction_model = ReductionModel()
    reduction_latency = profiler.measure_latency(reduction_model, input_tensor, iterations=20)
    reduction_flops = size * 32  # Sum reduction

    operations.append({
        'operation': 'Reduction',
        'latency_ms': reduction_latency,
        'flops': reduction_flops,
        'gflops_per_second': (reduction_flops / 1e9) / (reduction_latency / 1000),
        'efficiency_class': 'memory-bound',
        'optimization_focus': 'parallelization'
    })

    print("\nOperation Efficiency Comparison:")
    print("Operation\t\tLatency(ms)\tGFLOP/s\t\tEfficiency Class\tOptimization Focus")
    print("-" * 95)

    for op in operations:
        print(f"{op['operation']:<15}\t{op['latency_ms']:.3f}\t\t"
              f"{op['gflops_per_second']:.2f}\t\t{op['efficiency_class']:<15}\t{op['optimization_focus']}")

    print("\n💡 Operation Optimization Insights:")

    # Find most and least efficient
    best_op = max(operations, key=lambda x: x['gflops_per_second'])
    worst_op = min(operations, key=lambda x: x['gflops_per_second'])

    print(f"Most efficient: {best_op['operation']} ({best_op['gflops_per_second']:.2f} GFLOP/s)")
    print(f"Least efficient: {worst_op['operation']} ({worst_op['gflops_per_second']:.2f} GFLOP/s)")

    # Count operation types
    memory_bound_ops = [op for op in operations if op['efficiency_class'] == 'memory-bound']
    compute_bound_ops = [op for op in operations if op['efficiency_class'] == 'compute-bound']

    print(f"\n🚀 Optimization Priority:")
    if len(memory_bound_ops) > len(compute_bound_ops):
        print("Focus on memory optimization: data locality, bandwidth, caching")
    else:
        print("Focus on compute optimization: better algorithms, vectorization")

def analyze_profiling_overhead():
    """📊 Measure the overhead of profiling itself."""
    print("\n📊 Analyzing Profiling Overhead...")

    # Test with and without profiling
    test_tensor = Tensor(np.random.randn(100, 100))
    iterations = 50

    # Without profiling - baseline measurement
    start_time = time.perf_counter()
    for _ in range(iterations):
        _ = test_tensor.data.copy()  # Simple operation
    end_time = time.perf_counter()
    baseline_ms = (end_time - start_time) * 1000

    # With profiling - includes measurement overhead
    profiler = Profiler()
    # Create a simple model for profiling overhead measurement
    class TestModel:
        def forward(self, x):
            return x + 1.0

    test_model = TestModel()
    start_time = time.perf_counter()
    for _ in range(iterations):
        _ = profiler.measure_latency(test_model, test_tensor, warmup=1, iterations=1)
    end_time = time.perf_counter()
    profiled_ms = (end_time - start_time) * 1000

    overhead_factor = profiled_ms / max(baseline_ms, 0.001)

    print(f"\nProfiling Overhead Analysis:")
    print(f"Baseline execution: {baseline_ms:.2f} ms")
    print(f"With profiling: {profiled_ms:.2f} ms")
    print(f"Profiling overhead: {overhead_factor:.1f}× slower")

    print(f"\n💡 Profiling Overhead Insights:")
    if overhead_factor < 2:
        print("Low overhead - suitable for frequent profiling")
    elif overhead_factor < 10:
        print("Moderate overhead - use for development and debugging")
    else:
        print("High overhead - use sparingly in production")

# Run optimization analysis
if __name__ == "__main__":
    benchmark_operation_efficiency()
    analyze_profiling_overhead()

📊 Benchmarking Operation Efficiency...

Operation Efficiency Comparison:
Operation		Latency(ms)	GFLOP/s		Efficiency Class	Optimization Focus
-----------------------------------------------------------------------------------------------
Elementwise    	0.003		2.77		memory-bound   	data_locality
Matrix Multiply	6.034		0.02		compute-bound  	algorithms
Reduction      	0.003		2.40		memory-bound   	parallelization

💡 Operation Optimization Insights:
Most efficient: Elementwise (2.77 GFLOP/s)
Least efficient: Matrix Multiply (0.02 GFLOP/s)

🚀 Optimization Priority:
Focus on memory optimization: data locality, bandwidth, caching

📊 Analyzing Profiling Overhead...

Profiling Overhead Analysis:
Baseline execution: 1.22 ms
With profiling: 1.97 ms
Profiling overhead: 1.6× slower

💡 Profiling Overhead Insights:
Low overhead - suitable for frequent profiling


In [41]:
def test_module():
    """🧪 Module Test: Complete Integration

    Comprehensive test of entire profiling module functionality.

    This final test runs before module summary to ensure:
    - All unit tests pass
    - Functions work together correctly
    - Module is ready for integration with TinyTorch
    """
    print("🧪 RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests (helpers first, then composition functions)
    print("Running helper unit tests...")
    test_unit_count_layer_parameters()
    test_unit_count_linear_flops()
    test_unit_count_conv_flops()
    test_unit_count_sequential_flops()
    test_unit_calculate_parameter_memory()
    test_unit_calculate_memory_efficiency()
    test_unit_compute_derived_metrics()
    test_unit_analyze_bottleneck()
    test_unit_estimate_backward_costs()
    test_unit_estimate_optimizer_memory()

    print("\nRunning composition unit tests...")
    test_unit_helper_functions()
    test_unit_parameter_counting()
    test_unit_flop_counting()
    test_unit_memory_measurement()
    test_unit_latency_measurement()
    test_unit_advanced_profiling()

    print("\nRunning integration scenarios...")

    # Test realistic usage patterns
    print("🧪 Integration Test: Complete Profiling Workflow...")

    # Create profiler
    profiler = Profiler()

    # Create test model and data
    from tinytorch.core.layers import Linear
    test_model = Linear(16, 32)
    test_input = Tensor(np.random.randn(8, 16))

    # Run complete profiling workflow
    print("1. Measuring model characteristics...")
    params = profiler.count_parameters(test_model)
    flops = profiler.count_flops(test_model, test_input.shape)
    memory = profiler.measure_memory(test_model, test_input.shape)
    latency = profiler.measure_latency(test_model, test_input, warmup=2, iterations=5)

    print(f"   Parameters: {params}")
    print(f"   FLOPs: {flops}")
    print(f"   Memory: {memory['peak_memory_mb']:.2f} MB")
    print(f"   Latency: {latency:.2f} ms")

    # Test advanced profiling
    print("2. Running advanced profiling...")
    forward_profile = profiler.profile_forward_pass(test_model, test_input)
    backward_profile = profiler.profile_backward_pass(test_model, test_input)

    assert 'gflops_per_second' in forward_profile
    assert 'total_latency_ms' in backward_profile
    print(f"   Forward GFLOP/s: {forward_profile['gflops_per_second']:.2f}")
    print(f"   Training latency: {backward_profile['total_latency_ms']:.2f} ms")

    # Test bottleneck analysis
    print("3. Analyzing performance bottlenecks...")
    bottleneck = forward_profile['bottleneck']
    efficiency = forward_profile['computational_efficiency']
    print(f"   Bottleneck: {bottleneck}")
    print(f"   Compute efficiency: {efficiency:.3f}")

    # Validate end-to-end workflow
    assert params >= 0, "Parameter count should be non-negative"
    assert flops >= 0, "FLOP count should be non-negative"
    assert memory['peak_memory_mb'] >= 0, "Memory usage should be non-negative"
    assert latency >= 0, "Latency should be non-negative"
    assert forward_profile['gflops_per_second'] >= 0, "GFLOP/s should be non-negative"
    assert backward_profile['total_latency_ms'] >= 0, "Total latency should be non-negative"
    assert bottleneck in ['memory', 'compute'], "Bottleneck should be memory or compute"
    assert 0 <= efficiency <= 1, "Efficiency should be between 0 and 1"

    print("✅ End-to-end profiling workflow works!")

    # Test production-like scenario
    print("4. Testing production profiling scenario...")

    # Simulate larger model analysis
    from tinytorch.core.layers import Linear
    large_model = Linear(512, 256)
    large_input = Tensor(np.random.randn(32, 512))  # Larger model input
    large_profile = profiler.profile_forward_pass(large_model, large_input)

    # Verify profile contains optimization insights
    assert 'bottleneck' in large_profile, "Profile should identify bottlenecks"
    assert 'memory_bandwidth_mbs' in large_profile, "Profile should measure memory bandwidth"

    print(f"   Large model analysis: {large_profile['bottleneck']} bottleneck")
    print(f"   Memory bandwidth: {large_profile['memory_bandwidth_mbs']:.1f} MB/s")

    print("✅ Production profiling scenario works!")

    print("\n" + "=" * 50)
    print("🎉 ALL TESTS PASSED! Module ready for export.")
    print("Run: tito module complete 14")

# Run comprehensive module test
if __name__ == "__main__":
    test_module()

🧪 RUNNING MODULE INTEGRATION TEST
Running helper unit tests...
🧪 Unit Test: _count_layer_parameters...
✅ Layer with bias: 55 parameters
✅ Layer without bias: 32 parameters
✅ No weight attribute: 0 parameters
✅ _count_layer_parameters works correctly!
🧪 Unit Test: _count_linear_flops...
✅ Linear(128, 64): 16384 FLOPs
✅ Linear(256, 256): 131072 FLOPs
✅ Batch-independent FLOPs confirmed
✅ _count_linear_flops works correctly!
🧪 Unit Test: _count_conv_flops...
✅ Conv2d(3, 16, 3): 884736 FLOPs
✅ Conv2d(3, 64, 7, stride=2): 236027904 FLOPs
✅ Missing attributes returns 0
✅ _count_conv_flops works correctly!
🧪 Unit Test: _count_sequential_flops...
✅ Sequential(128->64->10): 17664 FLOPs
✅ Single-layer sequential: 1024 FLOPs
✅ _count_sequential_flops works correctly!
🧪 Unit Test: _calculate_parameter_memory...
✅ 1M params = 4.0 MB
✅ Empty model = 0.0 MB
✅ _calculate_parameter_memory works correctly!
🧪 Unit Test: _calculate_memory_efficiency...
✅ Perfect efficiency: 1.0
✅ Half efficiency: 0.5
✅ Cl

In [42]:
def demo_profiling():
    """🎯 See your profiler reveal model secrets."""
    print("🎯 AHA MOMENT: Know Your Model")
    print("=" * 45)

    # Create a simple model
    layer = Linear(784, 128)

    # Profile it
    profiler = Profiler()
    params = profiler.count_parameters(layer)
    flops = profiler.count_flops(layer, input_shape=(1, 784))

    print(f"Model: Linear(784 → 128)")
    print(f"\nParameters: {params:,}")
    print(f"  = 784 × 128 weights + 128 biases")

    print(f"\nFLOPs: {flops:,}")
    print(f"  = 784 × 128 × 2 (multiply-add per output)")

    print(f"\nMemory: {params * 4 / 1024:.1f} KB (at FP32)")

    print("\n✨ Profiling reveals optimization opportunities!")

In [43]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_profiling()

🧪 RUNNING MODULE INTEGRATION TEST
Running helper unit tests...
🧪 Unit Test: _count_layer_parameters...
✅ Layer with bias: 55 parameters
✅ Layer without bias: 32 parameters
✅ No weight attribute: 0 parameters
✅ _count_layer_parameters works correctly!
🧪 Unit Test: _count_linear_flops...
✅ Linear(128, 64): 16384 FLOPs
✅ Linear(256, 256): 131072 FLOPs
✅ Batch-independent FLOPs confirmed
✅ _count_linear_flops works correctly!
🧪 Unit Test: _count_conv_flops...
✅ Conv2d(3, 16, 3): 884736 FLOPs
✅ Conv2d(3, 64, 7, stride=2): 236027904 FLOPs
✅ Missing attributes returns 0
✅ _count_conv_flops works correctly!
🧪 Unit Test: _count_sequential_flops...
✅ Sequential(128->64->10): 17664 FLOPs
✅ Single-layer sequential: 1024 FLOPs
✅ _count_sequential_flops works correctly!
🧪 Unit Test: _calculate_parameter_memory...
✅ 1M params = 4.0 MB
✅ Empty model = 0.0 MB
✅ _calculate_parameter_memory works correctly!
🧪 Unit Test: _calculate_memory_efficiency...
✅ Perfect efficiency: 1.0
✅ Half efficiency: 0.5
✅ Cl